# BirdCLEF+ 2026 — Submission (v3/v4: MLP probe + attn temporal + student + mel-CNN ensemble + TopN)

Thin inference notebook. All training happens locally or on Kaggle GPU; this notebook only:

1. Loads a **single artifact bundle** (priors + probe + temporal + student + post-proc flags + per-class delta) from a private Kaggle dataset. The bundle's `probe_type` / `temporal_type` tags decide which head arithmetic to run at inference — both Stage 1 (`linear` / `lite`) and Stage 2 (`mlp` / `attn`) bundles are supported.
2. Runs **Perch v2** on every test soundscape, projecting 10k-class logits onto the 234 competition classes.
3. *(v4, optional)* Runs **3-seed MobileNetV3-Small mel-CNN** on the same raw 5 s waveforms — ONNX runtime, CPU. The CNN was distilled from the Stage-1 pseudo-teacher on 127 k cross-site windows, so it fills the site-shift gap (by-file OOF 0.9495 vs by-site OOF 0.8477).
4. Fuses perch + prior + probe + temporal + α·student + α·cnn on each window (weights come from the bundle — **no hardcoding**). Stage 2 adds a per-class logit shift (`delta_per_class`) — zeros by default, reserved for Stage 3 rank-aware ensemble.
5. Applies the OOF-picked post-processing recipe from the bundle (TopN=1 / isotonic / both, in the winning order).
6. Writes `submission.csv`.

Runtime policy: **no active truncation and no CNN auto-drop**. The previous 87-min safety stop caused PB 0.802 in v4.1 by silently leaving ~20% of files out of `submission.csv`. We now process every test file end-to-end with the full ensemble (Perch + priors + heads + α·student + α·CNN). If we ever exceed Kaggle's real CPU cap we shrink inference cost (batch size, CNN toggle, ensemble size), never truncate files.

Cost tuning for Kaggle CPU:
- Prefetch audio decode overlapped with Perch inference
- `batch_files=16` (matches proven PB 0.924 baseline)
- Explicit TF thread pinning (inter=2 intra=4)
- XLA warmup pre-pass matched to the inference batch shape (16×12 windows)
- CNN fp16 ONNX, 3 seeds, `intra_op_num_threads=4` / `inter_op_num_threads=2`

In [ ]:
# Cell 0 — Install TF 2.20 (required for Perch v2 StableHLO on Kaggle CPU)
!pip install -q --no-deps /kaggle/input/notebooks/ashok205/tf-wheels/tf_wheels/tensorboard-2.20.0-py3-none-any.whl
!pip install -q --no-deps /kaggle/input/notebooks/ashok205/tf-wheels/tf_wheels/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

# Install onnxruntime for the mel-CNN ensemble (v4). Kaggle default image doesn't ship it,
# and Code Competition submissions run with Internet=OFF, so we pull wheels bundled in
# the 'birdclef-2026' dataset. --no-deps keeps Kaggle's preinstalled numpy/protobuf/flatbuffers intact.
import glob as _g
_wheel_dirs = [
    '/kaggle/input/datasets/tiantanghuaxiao/birdclef-2026/wheels',
    '/kaggle/input/birdclef-2026/wheels',
]
_ort_wheel = None
for _d in _wheel_dirs:
    _matches = sorted(_g.glob(_d + '/onnxruntime-*.whl'))
    if _matches:
        _ort_wheel = _matches[0]
        break
if _ort_wheel is not None:
    print(f'installing onnxruntime from {_ort_wheel}')
    !pip install -q --no-deps {_ort_wheel}
else:
    print(f'WARN: onnxruntime wheel not found under {_wheel_dirs}; CNN ensemble will be disabled.')

In [ ]:
# Cell 1 — Imports and config
import os, gc, re, time, pickle
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "2")
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "4")

import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
from tqdm.auto import tqdm

try:
    tf.config.threading.set_inter_op_parallelism_threads(2)
    tf.config.threading.set_intra_op_parallelism_threads(4)
except RuntimeError:
    pass

tf.experimental.numpy.experimental_enable_numpy_behavior()

_WALL_START = time.time()

BASE = Path("/kaggle/input/competitions/birdclef-2026")
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")
BUNDLE_PATH = Path("/kaggle/input/datasets/tiantanghuaxiao/birdclef-2026/submission_bundle.pkl")

SR = 32000
WINDOW_SEC = 5
FILE_SAMPLES = 60 * SR
WINDOW_SAMPLES = SR * WINDOW_SEC
N_WINDOWS = 12

# No active truncation. Kaggle's real CPU code-competition cap is ~9h; we let the full test
# set run end-to-end so submission.csv is never missing rows. If the run genuinely
# exceeds Kaggle's hard limit we'll address it by shrinking inference cost (batch size,
# CNN toggle), not by truncating files mid-way (previous 87-min cap caused PB 0.802).
TIME_BUDGET_SEC = int(9 * 60 * 60)  # 9h sentinel; code below never early-exits on it.

CFG = {
    "batch_files": 16,                 # matches pantanal PB 0.924 reference; proven to fit 90 min
    "gc_every": 8,
    "verbose": False,
    "proxy_reduce": "max",
    "topn": 1,
    "prefetch": True,                  # decode next batch while Perch crunches current
    "time_budget_sec": TIME_BUDGET_SEC,
    "log_every_batches": 10,
}
# CNN ensemble cost control (module-level so Cell 3 sees it before CFG is used).
# v4.2 (3 seeds) hit Kaggle 9h Notebook Timeout on the hidden test. Single seed saves
# ~2/3 of CNN inference cost; our alpha sweep showed α_cnn stable across 1.25-1.75,
# so the 3-seed signal is largely redundant. Raise back to 3 once timing proven safe.

print("TF:", tf.__version__)
print("BASE exists :", BASE.exists())
print("MODEL_DIR   :", MODEL_DIR.exists())
print("BUNDLE exists:", BUNDLE_PATH.exists())
print("TF threads  : inter=2 intra=4")

In [ ]:
# Cell 2 — Load artifact bundle (v1/v2: linear probe + lite temporal; v3: MLP probe + attn temporal)
with BUNDLE_PATH.open("rb") as f:
    bundle = pickle.load(f)

PRIMARY_LABELS = bundle["primary_labels"]
N_CLASSES = len(PRIMARY_LABELS)

# Mapping
MAPPED_POS = bundle["mapping"]["mapped_pos"].astype(np.int32)
MAPPED_BC_INDICES = bundle["mapping"]["mapped_bc_indices"].astype(np.int32)
PROXY_POS_TO_BC = bundle["mapping"]["proxy_pos_to_bc"]

# Priors
priors = bundle["priors"]
global_prob = priors["global_prob"]
site_map = dict(zip(priors["site_keys"], priors["site_vals"]))
hour_map = {int(k): v for k, v in zip(priors["hour_keys"], priors["hour_vals"])}
month_map = {int(k): v for k, v in zip(priors["month_keys"], priors["month_vals"])}
site_hour_map = {tuple(k): v for k, v in zip(priors["site_hour_keys"], priors["site_hour_vals"])}

# Embedding pipeline (scaler + PCA) — always present; used by student MLP and
# Stage-1 linear probe. Stage-2 MLP probe uses raw embeddings.
pipe = bundle["embedding_pipeline"]
SCALER_MEAN = pipe["scaler_mean"]; SCALER_SCALE = pipe["scaler_scale"]
PCA_MEAN = pipe["pca_mean"]; PCA_COMP = pipe["pca_components"]

# ---- Probe head dispatch -----------------------------------------------------
probe = bundle["probe"]
PROBE_TYPE = str(bundle.get("probe_type", probe.get("probe_type", "linear")))
PROBE_ACTIVE = probe["active_class_idx"].astype(np.int32)

if PROBE_TYPE == "mlp":
    # 1536 -> hidden -> K MLP; operates on RAW perch embeddings
    PROBE_W0 = probe["w0"]; PROBE_B0 = probe["b0"]
    PROBE_W1 = probe["w1"]; PROBE_B1 = probe["b1"]
    PROBE_COEF = None; PROBE_BIAS = None
else:
    # Stage-1 linear probe: Xpca @ coef.T + intercept
    PROBE_COEF = probe["coef_mat"]; PROBE_BIAS = probe["intercept_vec"]
    PROBE_W0 = PROBE_B0 = PROBE_W1 = PROBE_B1 = None

# ---- Temporal head dispatch --------------------------------------------------
temp = bundle["temporal"]
TEMP_TYPE = str(bundle.get("temporal_type", temp.get("temporal_type", "lite")))
TEMP_ACTIVE = temp["active_class_idx"].astype(np.int32)

if TEMP_TYPE == "attn":
    # 1-layer transformer over the 12-window raw embedding sequence
    TATTN = temp
    TATTN_D = int(temp["d_model"])
    TATTN_H = int(temp["n_heads"])
    TATTN_T = int(temp["window_per_file"])
    TEMP_STATS = None; TEMP_COEF = None; TEMP_BIAS = None
else:
    # Stage-1 temporal-lite: 4 hand-picked stats × LogReg
    TEMP_STATS = tuple(temp["stats"])
    TEMP_COEF = temp["coef_mat"]
    TEMP_BIAS = temp["intercept_vec"]
    TATTN = None

# ---- Fusion weights ----------------------------------------------------------
FW = bundle["fusion_weights"]
A_PERCH, A_PRIOR, A_PROBE, A_TEMP = (
    float(FW["alpha_perch"]), float(FW["alpha_prior"]),
    float(FW["alpha_probe"]), float(FW["alpha_temp"]),
)

# ---- Optional student / calibration ------------------------------------------
STUDENT = bundle.get("student")
CALIBRATION = bundle.get("calibration")

STUDENT_ACTIVE = None
STUDENT_W0 = STUDENT_B0 = STUDENT_W1 = STUDENT_B1 = STUDENT_W2 = STUDENT_B2 = None
ALPHA_STUDENT = 0.0
if STUDENT is not None:
    ALPHA_STUDENT = float(STUDENT.get("alpha_student", 0.0))
    STUDENT_ACTIVE = STUDENT["active_class_idx"].astype(np.int32)
    STUDENT_W0 = STUDENT["w0"]; STUDENT_B0 = STUDENT["b0"]
    STUDENT_W1 = STUDENT["w1"]; STUDENT_B1 = STUDENT["b1"]
    STUDENT_W2 = STUDENT["w2"]; STUDENT_B2 = STUDENT["b2"]

# ---- Post-processing + per-class delta ---------------------------------------
POSTPROC = bundle.get("postprocess", {})
APPLY_TOPN1    = bool(POSTPROC.get("apply_topn1", True))
APPLY_ISOTONIC = bool(POSTPROC.get("apply_isotonic", False))
TOPN_FIRST     = bool(POSTPROC.get("topn_first", False))
DELTA_PER_CLASS = bundle.get("delta_per_class")
if DELTA_PER_CLASS is None:
    DELTA_PER_CLASS = np.zeros(N_CLASSES, dtype=np.float32)
else:
    DELTA_PER_CLASS = DELTA_PER_CLASS.astype(np.float32)

print("bundle tag:", bundle.get("tag"))
print("n_classes :", N_CLASSES)
print("n_mapped  :", len(MAPPED_POS), " n_proxy:", len(PROXY_POS_TO_BC))
print("probe_type:", PROBE_TYPE, " active:", PROBE_ACTIVE.shape)
print("temp_type :", TEMP_TYPE,  " active:", TEMP_ACTIVE.shape)
print("fusion    :", {k: round(v, 3) for k, v in FW.items()},
      "  alpha_student:", round(ALPHA_STUDENT, 3))
print("postproc  : topn1=", APPLY_TOPN1, " iso=", APPLY_ISOTONIC,
      " topn_first=", TOPN_FIRST)
print("delta_per_class: min/max/mean =",
      round(float(DELTA_PER_CLASS.min()), 3),
      round(float(DELTA_PER_CLASS.max()), 3),
      round(float(DELTA_PER_CLASS.mean()), 3))
print("student/calibration:", STUDENT is not None, "/", CALIBRATION is not None)

# --------------------------------------------------------------------------
# Mel-CNN ensemble (v4): optional, auto-discovers ONNX models under Kaggle inputs.
# Bundle key `cnn` controls alpha_cnn; if missing or 0, CNN is disabled even if
# ONNX files exist. Wall-time guard in Cell 7 may also disable at runtime.
# --------------------------------------------------------------------------
CNN_CFG = bundle.get("cnn", {}) or {}
ALPHA_CNN = float(CNN_CFG.get("alpha_cnn", 0.0))

# Seed selection policy, data-driven from a local per-seed sweep on 708 labeled rows,
# sweeping alpha_cnn independently per configuration (best AUC by by_site):
#   1-seed 215 @ a=0.50 (by_site=0.9151)   -0.017 vs 2-seed  <-- v4.4 pick
#   1-seed 322 @ a=1.00 (by_site=0.9053)   -0.027 vs 2-seed
#   2-seed [215+322] @ a=1.50 (by_site=0.9323)                baseline v4.3 (TIMED OUT)
#   3-seed ensemble  @ a=1.50 (by_site=0.9238)                v4.2 (TIMED OUT)
# Both v4.2 and v4.3 hit Kaggle Notebook Timeout. Dropping to 1 seed cuts CNN cost
# from ~1.4 s/file to ~0.67 s/file and should fit in the wall-clock budget.
# Seed 215 wins single-seed because its optimal alpha is lower (calibrated better).
CNN_MAX_SEEDS = 1
# Prefer-list: these substrings are matched against the ONNX stem in priority order.
# First CNN_MAX_SEEDS that exist on disk are used. Anything not matching is skipped.
# Seed 20260101 is excluded from practical use (overconfident mean=0.158, hurts ensemble).
CNN_SEED_PRIORITY = ["20260215", "20260322", "20260101"]

_CNN_SEARCH_DIRS = [
    Path("/kaggle/input/datasets/tiantanghuaxiao/birdclef-2026-cnn"),
    Path("/kaggle/input/datasets/tiantanghuaxiao/birdclef-2026"),
    Path("/kaggle/input/datasets/tiantanghuaxiao/birdclef-2026-distill"),
    Path("/kaggle/input/birdclef-2026-cnn"),
    Path("/kaggle/input/birdclef-2026"),
]
_onnx_candidates: list[Path] = []
for d in _CNN_SEARCH_DIRS:
    if d.exists():
        # Prefer *_fp16.onnx if both are present
        fp16 = sorted(d.glob("mel_cnn_seed*_fp16.onnx"))
        if fp16:
            _onnx_candidates.extend(fp16)
        else:
            _onnx_candidates.extend(sorted(d.glob("mel_cnn_seed*.onnx")))
# De-dup by stem preferring fp16
_by_stem = {}
for p in _onnx_candidates:
    stem = p.stem.replace("_fp16", "")
    if stem not in _by_stem or p.stem.endswith("_fp16"):
        _by_stem[stem] = p
# Data-driven seed selection: iterate CNN_SEED_PRIORITY, pick the first
# CNN_MAX_SEEDS seeds that are present on disk. Everything else falls back to
# alphabetical order so we still produce a stable ordering for unknown seeds.
_prio_map = {}
for rank, substr in enumerate(CNN_SEED_PRIORITY):
    for stem, p in _by_stem.items():
        if substr in stem and stem not in _prio_map:
            _prio_map[stem] = (rank, p)
            break
_other = [(len(CNN_SEED_PRIORITY) + i, p) for i, (stem, p) in
          enumerate(sorted(_by_stem.items())) if stem not in _prio_map]
_ranked = list(_prio_map.values()) + _other
_ranked.sort(key=lambda t: t[0])
CNN_ONNX_PATHS = [p for _, p in _ranked]
if CNN_MAX_SEEDS > 0:
    CNN_ONNX_PATHS = CNN_ONNX_PATHS[:CNN_MAX_SEEDS]

CNN_SESSIONS = []
CNN_INPUT_NAME = None
CNN_WINDOW_LEN = 5 * SR  # must match training (CFG.window_len = 160000)
if ALPHA_CNN > 0 and len(CNN_ONNX_PATHS) > 0:
    try:
        import onnxruntime as ort  # type: ignore
        sess_opts = ort.SessionOptions()
        sess_opts.intra_op_num_threads = 4
        sess_opts.inter_op_num_threads = 2
        sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        for p in CNN_ONNX_PATHS:
            s = ort.InferenceSession(str(p), sess_options=sess_opts,
                                      providers=["CPUExecutionProvider"])
            CNN_SESSIONS.append(s)
            if CNN_INPUT_NAME is None:
                CNN_INPUT_NAME = s.get_inputs()[0].name
        print(f"CNN ensemble: {len(CNN_SESSIONS)} seeds loaded "
              f"(alpha_cnn={ALPHA_CNN:.3f}, input={CNN_INPUT_NAME})")
        for p in CNN_ONNX_PATHS:
            sz = p.stat().st_size / (1024 * 1024)
            print(f"    {p.name}  {sz:.2f} MB")
    except Exception as e:
        print("CNN ensemble: failed to init onnxruntime:", repr(e))
        CNN_SESSIONS = []
elif ALPHA_CNN > 0:
    print(f"CNN ensemble: alpha_cnn={ALPHA_CNN:.3f} but no ONNX found in {_CNN_SEARCH_DIRS}")
else:
    print(f"CNN ensemble: disabled (alpha_cnn={ALPHA_CNN:.3f})")

CNN_ENABLED = len(CNN_SESSIONS) > 0 and ALPHA_CNN > 0

In [ ]:
# Cell 3 — Load Perch v2
birdclassifier = tf.saved_model.load(str(MODEL_DIR))
infer_fn = birdclassifier.signatures["serving_default"]
print("Perch loaded.")

In [ ]:
# Cell 4a — Perch XLA warmup
# Match the exact batch shape used in inference so XLA JIT compiles once.
_t0 = time.time()
_dummy_rows = 16 * N_WINDOWS          # must equal CFG["batch_files"] * N_WINDOWS
_dummy = np.zeros((_dummy_rows, WINDOW_SAMPLES), dtype=np.float32)
_ = infer_fn(inputs=tf.convert_to_tensor(_dummy))
print(f"Perch warmup: {time.time() - _t0:.1f}s  (shape {_dummy.shape})")
del _dummy, _
gc.collect()

In [ ]:
# Cell 4 — Helpers
FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_soundscape_filename(name):
    m = FNAME_RE.match(name)
    if not m:
        return {"site": None, "hour_utc": -1, "month": -1}
    _, site, ymd, hms = m.groups()
    dt = pd.to_datetime(ymd, format="%Y%m%d", errors="coerce")
    return {"site": site, "hour_utc": int(hms[:2]), "month": int(dt.month) if pd.notna(dt) else -1}

def read_soundscape_60s(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2: y = y.mean(axis=1)
    if sr != SR: raise ValueError(f"bad sr {sr}")
    if len(y) < FILE_SAMPLES: y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    elif len(y) > FILE_SAMPLES: y = y[:FILE_SAMPLES]
    return y

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

EPS = 1e-4
def logit_clip(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))

def apply_embedding_pipeline(emb):
    scaled = (emb.astype(np.float32) - SCALER_MEAN[None, :]) / SCALER_SCALE[None, :]
    centered = scaled - PCA_MEAN[None, :]
    return (centered @ PCA_COMP.T).astype(np.float32)

_GLOBAL_PROB_F32 = global_prob.astype(np.float32)

def build_prior_logits(meta_df):
    # vectorized: look up each key group, add via running sum/count then divide
    n = len(meta_df)
    acc = np.tile(_GLOBAL_PROB_F32, (n, 1))        # (n, C)
    cnt = np.ones(n, dtype=np.float32)

    sites = meta_df["site"].to_numpy()
    hours = meta_df["hour_utc"].to_numpy().astype(np.int64)
    months = meta_df["month"].to_numpy().astype(np.int64)

    for i in range(n):
        s, h, m = sites[i], int(hours[i]), int(months[i])
        k = 0
        if s in site_map:
            acc[i] += site_map[s]; k += 1
        if h in hour_map:
            acc[i] += hour_map[h]; k += 1
        if m in month_map:
            acc[i] += month_map[m]; k += 1
        if (s, h) in site_hour_map:
            acc[i] += site_hour_map[(s, h)]; k += 1
        if k:
            cnt[i] = 1.0 + k

    acc /= cnt[:, None]
    return logit_clip(acc).astype(np.float32)

def compute_temp_features(seq_scores, stat_name):
    if stat_name == "mean":  return seq_scores.mean(axis=1)
    if stat_name == "max":   return seq_scores.max(axis=1)
    if stat_name == "std":   return seq_scores.std(axis=1)
    if stat_name == "delta": return seq_scores[:, -1, :] - seq_scores[:, 0, :]
    if stat_name == "p90":   return np.quantile(seq_scores, 0.9, axis=1)
    raise ValueError(stat_name)

# ---- Stage 2 MLP probe forward (raw emb -> K active logits) ------------------
def apply_mlp_probe(emb_raw):
    h = emb_raw @ PROBE_W0 + PROBE_B0
    np.maximum(h, 0.0, out=h)
    return (h @ PROBE_W1 + PROBE_B1).astype(np.float32)

# ---- Stage 2 attention temporal forward (emb_seq -> K file-level logits) -----
def _softmax_np(x, axis):
    m = x.max(axis=axis, keepdims=True)
    e = np.exp(x - m)
    return e / e.sum(axis=axis, keepdims=True)

def _layernorm_np(x, gamma, beta, eps=1e-5):
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps) * gamma + beta

def apply_temporal_attn(emb_seq):
    """emb_seq: (F, T, 1536) -> (F, K) file-level logits over active classes."""
    art = TATTN
    d = TATTN_D
    H = TATTN_H
    d_k = d // H
    x = emb_seq @ art["w_in"] + art["b_in"]
    x = x + art["pos_emb"][None, :, :]

    h = _layernorm_np(x, art["g1"], art["b1"])
    qkv = h @ art["w_qkv"] + art["b_qkv"]
    q, k_, v = np.split(qkv, 3, axis=-1)
    F_, T_, _ = q.shape
    q = q.reshape(F_, T_, H, d_k).transpose(0, 2, 1, 3)
    k_ = k_.reshape(F_, T_, H, d_k).transpose(0, 2, 1, 3)
    v = v.reshape(F_, T_, H, d_k).transpose(0, 2, 1, 3)
    att = q @ k_.transpose(0, 1, 3, 2) / np.sqrt(d_k)
    att = _softmax_np(att.astype(np.float32), axis=-1)
    attn = att @ v
    attn = attn.transpose(0, 2, 1, 3).reshape(F_, T_, d)
    attn = attn @ art["w_out"] + art["b_out"]
    x = x + attn

    h = _layernorm_np(x, art["g2"], art["b2"])
    ff = h @ art["w_ff1"] + art["b_ff1"]
    np.maximum(ff, 0.0, out=ff)
    ff = ff @ art["w_ff2"] + art["b_ff2"]
    x = x + ff

    pooled = x.mean(axis=1)
    return (pooled @ art["w_cls"] + art["b_cls"]).astype(np.float32)

In [ ]:
# Cell 5 — Inference over test soundscapes (prefetch + time-budget safety)
# NOTE: no window skipping / energy gating. Every test window is scored by Perch.
from concurrent.futures import ThreadPoolExecutor

def _decode_batch(batch_paths):
    """Decode a batch of soundscape paths into (audio tensor, meta DataFrame)."""
    bn = len(batch_paths)
    nrows = bn * N_WINDOWS
    x = np.empty((nrows, WINDOW_SAMPLES), dtype=np.float32)
    row_ids = np.empty(nrows, dtype=object)
    filenames = np.empty(nrows, dtype=object)
    sites = np.empty(nrows, dtype=object)
    hours = np.empty(nrows, dtype=np.int16)
    months = np.empty(nrows, dtype=np.int16)
    w = 0
    for path in batch_paths:
        y = read_soundscape_60s(path)
        x[w:w+N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
        meta = parse_soundscape_filename(path.name)
        stem = path.stem
        row_ids[w:w+N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
        filenames[w:w+N_WINDOWS] = path.name
        sites[w:w+N_WINDOWS] = meta["site"]
        hours[w:w+N_WINDOWS] = int(meta["hour_utc"])
        months[w:w+N_WINDOWS] = int(meta["month"])
        w += N_WINDOWS
    meta_df = pd.DataFrame({
        "row_id": row_ids, "filename": filenames,
        "site": sites, "hour_utc": hours, "month": months,
    })
    return x, meta_df

EPS_LOGIT = 1e-4

def _cnn_ensemble_logits(x):
    """Run the 3-seed CNN on raw 5 s waveforms -> (nrows, N_CLASSES) logits.
    Returns None only if the ensemble is fully disabled (no ONNX or alpha_cnn==0).
    """
    if not CNN_ENABLED:
        return None
    # Sessions return sigmoid probabilities (ExportWrapper in train notebook).
    probs_sum = None
    for s in CNN_SESSIONS:
        p = s.run(None, {CNN_INPUT_NAME: x})[0].astype(np.float32, copy=False)
        probs_sum = p if probs_sum is None else probs_sum + p
    probs = probs_sum / float(len(CNN_SESSIONS))
    probs = np.clip(probs, EPS_LOGIT, 1.0 - EPS_LOGIT)
    return (np.log(probs) - np.log1p(-probs)).astype(np.float32)

def _fuse_batch(x, meta_df):
    """Full Perch + fusion on all rows of the batch.

    Dispatches on PROBE_TYPE (linear / mlp) and TEMP_TYPE (lite / attn).
    Optionally adds the 3-seed CNN ensemble (raw-waveform ONNX).
    """
    nrows = x.shape[0]
    batch_n_files = nrows // N_WINDOWS

    outputs = infer_fn(inputs=tf.convert_to_tensor(x))
    logits = outputs["label"].numpy().astype(np.float32, copy=False)
    emb = outputs["embedding"].numpy().astype(np.float32, copy=False)
    outputs = None

    # Project 10k-class Perch logits onto the 234 competition classes.
    scores_raw = np.zeros((nrows, N_CLASSES), dtype=np.float32)
    scores_raw[:, MAPPED_POS] = logits[:, MAPPED_BC_INDICES]
    for pos, bc_arr in PROXY_POS_TO_BC.items():
        sub = logits[:, bc_arr]
        scores_raw[:, pos] = sub.max(axis=1) if CFG["proxy_reduce"] == "max" else sub.mean(axis=1)

    prior_logits = build_prior_logits(meta_df)
    final = A_PERCH * scores_raw + A_PRIOR * prior_logits

    Xpca = apply_embedding_pipeline(emb)       # still needed for student MLP

    # ---- Probe head ---------------------------------------------------------
    # Short-circuit when A_PROBE==0 (e.g. v4 drops the MLP probe entirely).
    if A_PROBE != 0.0:
        if PROBE_TYPE == "mlp":
            probe_active_scores = apply_mlp_probe(emb)                 # raw 1536-D
        else:
            probe_active_scores = (Xpca @ PROBE_COEF.T + PROBE_BIAS[None, :]).astype(np.float32)
        final[:, PROBE_ACTIVE] += A_PROBE * probe_active_scores

    # ---- Student MLP (PCA input) -------------------------------------------
    if STUDENT is not None and ALPHA_STUDENT > 0.0:
        h = Xpca @ STUDENT_W0 + STUDENT_B0
        np.maximum(h, 0.0, out=h)
        h = h @ STUDENT_W1 + STUDENT_B1
        np.maximum(h, 0.0, out=h)
        student_logits = (h @ STUDENT_W2 + STUDENT_B2).astype(np.float32)
        final[:, STUDENT_ACTIVE] += ALPHA_STUDENT * student_logits

    # ---- Temporal head -----------------------------------------------------
    # Short-circuit when A_TEMP==0 (v4 drops the self-attn temporal head).
    if A_TEMP != 0.0:
        if TEMP_TYPE == "attn":
            emb_seq = emb.reshape(batch_n_files, N_WINDOWS, emb.shape[1])
            temp_file_logits = apply_temporal_attn(emb_seq)            # (F, K)
        else:
            seq = scores_raw.reshape(batch_n_files, N_WINDOWS, N_CLASSES)
            stats_per_class = np.stack(
                [compute_temp_features(seq, s)[:, TEMP_ACTIVE] for s in TEMP_STATS],
                axis=1,
            )
            temp_file_logits = np.einsum(
                "fsc,cs->fc", stats_per_class, TEMP_COEF
            ).astype(np.float32)
            temp_file_logits += TEMP_BIAS[None, :]
        temp_window_logits = np.repeat(temp_file_logits, N_WINDOWS, axis=0)
        final[:, TEMP_ACTIVE] += A_TEMP * temp_window_logits

    # ---- Mel-CNN ensemble (v4) ---------------------------------------------
    cnn_logits = _cnn_ensemble_logits(x)
    if cnn_logits is not None:
        # CNN output is already aligned with PRIMARY_LABELS class order.
        final = final + ALPHA_CNN * cnn_logits

    # ---- Per-class logit shift (Stage 2.3; zeros in v3, real values in v4) -
    final = final + DELTA_PER_CLASS[None, :]

    probs = sigmoid(final)
    df = pd.DataFrame(probs, columns=PRIMARY_LABELS)
    df.insert(0, "row_id", meta_df["row_id"].values)
    df.insert(1, "filename", meta_df["filename"].values)
    return df

# ---- driver ----
test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
is_hidden_test = len(test_paths) > 0
if not is_hidden_test:
    test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:20]
    print("(local sanity) using first 20 train_soundscapes")

bf = CFG["batch_files"]
batches = [test_paths[i:i+bf] for i in range(0, len(test_paths), bf)]
n_batches = len(batches)
print(f"test files: {len(test_paths)}  batches: {n_batches}  batch_files: {bf}")

parts = []
processed_files = 0
_loop_t0 = time.time()

decoder_pool = ThreadPoolExecutor(max_workers=1) if CFG["prefetch"] else None
next_future = decoder_pool.submit(_decode_batch, batches[0]) if decoder_pool else None

for batch_idx, batch_paths in enumerate(batches):
    # NO early exit and NO CNN auto-drop. Previous version truncated the file loop at 87 min
    # (PB 0.802 ← missing rows filled with 0). We now process every file with CNN always on;
    # if Kaggle's hard cap is hit we'll optimize cost rather than ship a partial submission.
    if decoder_pool is not None:
        x, meta_df = next_future.result()
        if batch_idx + 1 < n_batches:
            next_future = decoder_pool.submit(_decode_batch, batches[batch_idx + 1])
    else:
        x, meta_df = _decode_batch(batch_paths)

    df = _fuse_batch(x, meta_df)
    parts.append(df)
    processed_files += len(batch_paths)

    del x, meta_df, df
    if CFG["gc_every"] and (batch_idx + 1) % CFG["gc_every"] == 0:
        gc.collect()

    if (batch_idx + 1) % CFG["log_every_batches"] == 0 or batch_idx == 0:
        loop_elapsed = time.time() - _loop_t0
        per_batch = loop_elapsed / (batch_idx + 1)
        remaining = (n_batches - batch_idx - 1) * per_batch
        wall_elapsed = time.time() - _WALL_START
        cnn_tag = "cnn=on" if CNN_ENABLED else "cnn=off"
        print(f"[{batch_idx+1:4d}/{n_batches}] files={processed_files}  "
              f"batch_avg={per_batch:.1f}s  eta={remaining/60:.1f}min  "
              f"wall={wall_elapsed/60:.1f}min  {cnn_tag}")

if decoder_pool is not None:
    decoder_pool.shutdown(wait=False, cancel_futures=True)

pred_df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(
    columns=["row_id", "filename"] + list(PRIMARY_LABELS)
)
del parts; gc.collect()
print(f"inference rows: {len(pred_df)}  processed_files: {processed_files}/{len(test_paths)}  "
      f"(no early-exit; full file coverage)")

In [ ]:
# Cell 6 — Post-processing (TopN smoothing + optional isotonic calibration)
def topn_smoothing(df, label_cols, n=1):
    probs = df[label_cols].to_numpy(dtype=np.float32)
    filenames = df["filename"].to_numpy()
    out = probs.copy()
    file_to_rows = pd.Series(np.arange(len(df))).groupby(filenames).apply(np.asarray)
    for _fn, idx in file_to_rows.items():
        sub = probs[idx]
        if n <= 1:
            avg_top = sub.max(axis=0)
        else:
            k = min(n, sub.shape[0])
            avg_top = np.partition(sub, -k, axis=0)[-k:].mean(axis=0)
        out[idx] = sub * avg_top[None, :]
    result = df.copy()
    result[label_cols] = out
    return result

def apply_isotonic(df, label_cols, calibration):
    if calibration is None:
        return df
    probs = df[label_cols].to_numpy(dtype=np.float32)
    for j, knots in calibration["per_class"].items():
        j = int(j)
        probs[:, j] = np.interp(probs[:, j], knots["x"], knots["y"]).astype(np.float32)
    result = df.copy()
    result[label_cols] = probs
    return result

# Apply post-processing in the order chosen by the OOF A/B test.
_use_topn1 = APPLY_TOPN1
_use_iso = APPLY_ISOTONIC and CALIBRATION is not None and len(CALIBRATION.get("per_class", {})) > 0

if _use_topn1 and _use_iso:
    if TOPN_FIRST:
        pred_df = topn_smoothing(pred_df, PRIMARY_LABELS, n=CFG["topn"])
        pred_df = apply_isotonic(pred_df, PRIMARY_LABELS, CALIBRATION)
    else:
        pred_df = apply_isotonic(pred_df, PRIMARY_LABELS, CALIBRATION)
        pred_df = topn_smoothing(pred_df, PRIMARY_LABELS, n=CFG["topn"])
elif _use_topn1:
    pred_df = topn_smoothing(pred_df, PRIMARY_LABELS, n=CFG["topn"])
elif _use_iso:
    pred_df = apply_isotonic(pred_df, PRIMARY_LABELS, CALIBRATION)
print(f"post-proc done. topn1={_use_topn1} iso={_use_iso} topn_first={TOPN_FIRST}")

In [ ]:
# Cell 7 — Write submission.csv
sample_sub = pd.read_csv(BASE / "sample_submission.csv")

if is_hidden_test:
    submission = sample_sub[["row_id"]].merge(
        pred_df.drop(columns=["filename"]), on="row_id", how="left"
    )
    submission[PRIMARY_LABELS] = submission[PRIMARY_LABELS].fillna(0.0).astype(np.float32)
    assert submission.shape == sample_sub.shape
    assert submission["row_id"].tolist() == sample_sub["row_id"].tolist()
else:
    submission = pred_df.drop(columns=["filename"])

submission.to_csv("submission.csv", index=False)
print("submission rows:", len(submission))
print("wall time min :", round((time.time() - _WALL_START) / 60, 2))
print("nan count     :", int(submission.isna().sum().sum()))